In [1]:
!pip install -U langgraph==1.2.16
!pip install -U langgraph-checkpoint-sqlite

In [2]:
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.sqlite import SqliteSaver

In [3]:
# -----------------------------
# 1. Define State Schema
# -----------------------------
class SummarizerState(TypedDict):
    input: str
    summary: str

In [4]:
# -----------------------------
# 2. Initialize LLM
# -----------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [5]:
# -----------------------------
# 3. Define Node
# -----------------------------
def summarize_node(state: SummarizerState):
    text = state["input"]
    response = llm.invoke(f"Summarize this text in 3 sentences:\n\n{text}")
    return {"input": text, "summary": response.content}

In [6]:
# -----------------------------
# 4. Build Graph
# -----------------------------
graph = StateGraph(SummarizerState)
graph.add_node("summarize", summarize_node)
graph.set_entry_point("summarize")
graph.add_edge("summarize", END)

In [9]:
# -----------------------------
# 5. Add SQLite Checkpointing
# -----------------------------
# This will store workflow state in a local SQLite file
sqlite_saver = SqliteSaver.from_conn_string("sqlite:///langgraph.db")
app = graph.compile(checkpointer=sqlite_saver)

TypeError: Invalid checkpointer provided. Expected an instance of `BaseCheckpointSaver`, `True`, `False`, or `None`. Received _GeneratorContextManager. Pass a proper saver (e.g., InMemorySaver, AsyncPostgresSaver).

In [ ]:
# -----------------------------
# 6. Run Example
# -----------------------------
text_to_summarize = """
LangGraph is a runtime for building stateful, multi-agent workflows.
It models workflows as graphs with nodes and edges, supports loops, retries,
human-in-the-loop steps, and durable state persistence.
"""
# Each run gets a thread_id for checkpointing
result = app.invoke({"input": text_to_summarize}, config={"configurable": {"thread_id": "demo-thread"}})
print("Summary:", result["summary"])